# Emparejamiento bipartito

En un grafo bipartito con lados $L$ y $R$, un emparejamiento es un conjunto de aristas sin vértices en común. Un camino aumentante empieza y acaba en vértices libres alternando aristas fuera y dentro del emparejamiento; invertirlo añade una pareja, y el emparejamiento es máximo cuando no queda ninguno (teorema de Berge). Buscando un camino desde cada vértice de $L$ (algoritmo de Kuhn) cuesta $O(V \cdot E)$; Hopcroft-Karp aumenta por fases muchos caminos mínimos disjuntos a la vez y baja a $O(E \sqrt{V})$. El tamaño del emparejamiento máximo coincide con el del cubrimiento mínimo de vértices (teorema de König), y la versión con costes, la asignación, se resuelve con el algoritmo húngaro en $O(n^3)$.

## 1. Kuhn con traza de los caminos aumentantes

Unos voluntarios del Carnaval pueden cubrir ciertos puestos. Emparejarlos con caminos aumentantes y, cada vez que se encuentra uno, mostrar el camino alternante (voluntario → puesto → voluntario → …) que se invierte. Comprobar que el resultado es un emparejamiento válido.

In [1]:
import random
import sys
from collections import deque
from itertools import combinations, permutations

import networkx as nx
import numpy as np
from scipy.optimize import linear_sum_assignment

sys.setrecursionlimit(10_000)


def kuhn(izquierda, adyacencia, traza=False):
    pareja_der = {}  # puesto -> voluntario

    def aumenta(u, vistos):
        for v in adyacencia.get(u, ()):
            if v in vistos:
                continue
            vistos.add(v)
            if v not in pareja_der:
                pareja_der[v] = u
                return [u, v]
            resto = aumenta(pareja_der[v], vistos)
            if resto:
                pareja_der[v] = u
                return [u, v, *resto]
        return None

    for u in izquierda:
        camino = aumenta(u, set())
        if traza and camino:
            print(" → ".join(camino))
    return {u: v for v, u in pareja_der.items()}


puestos = {
    "Ana": ["escenario", "taquilla"],
    "Luis": ["escenario", "guardarropa"],
    "Eva": ["taquilla", "barra"],
    "Pau": ["barra", "megafonía"],
    "Noa": ["megafonía"],
}
emparejamiento = kuhn(puestos, puestos, traza=True)
print(emparejamiento)


def es_emparejamiento(m, adyacencia):
    return len(set(m.values())) == len(m) and all(v in adyacencia[u] for u, v in m.items())


assert es_emparejamiento(emparejamiento, puestos) and len(emparejamiento) == 5

Ana → escenario
Luis → escenario → Ana → taquilla
Eva → taquilla → Ana → escenario → Luis → guardarropa
Pau → barra
Noa → megafonía
{'Ana': 'escenario', 'Eva': 'taquilla', 'Luis': 'guardarropa', 'Pau': 'barra', 'Noa': 'megafonía'}


## 2. Contra fuerza bruta y networkx

Comprobar `kuhn` en grafos bipartitos aleatorios pequeños contra probar todos los subconjuntos de aristas, y en grafos medianos contra `nx.bipartite.maximum_matching`.

In [2]:
def emparejamiento_bruto(izquierda, adyacencia):
    aristas = [(u, v) for u in izquierda for v in adyacencia.get(u, ())]
    for k in range(min(len(izquierda), len(aristas)), 0, -1):
        for elegidas in combinations(aristas, k):
            if len({u for u, _ in elegidas}) == len({v for _, v in elegidas}) == k:
                return k
    return 0


def aleatorio(n_izq, n_der, p, rng):
    izq = [f"i{k}" for k in range(n_izq)]
    return izq, {u: [f"d{k}" for k in range(n_der) if rng.random() < p] for u in izq}


rnd = random.Random(3)
for _ in range(300):
    izq, ady = aleatorio(rnd.randint(1, 5), rnd.randint(1, 5), 0.4, rnd)
    m = kuhn(izq, ady)
    assert es_emparejamiento(m, ady) and len(m) == emparejamiento_bruto(izq, ady)


def a_networkx(izq, ady):
    g = nx.Graph()
    g.add_nodes_from(izq)
    g.add_edges_from((u, v) for u in izq for v in ady[u])
    return g


for _ in range(50):
    izq, ady = aleatorio(rnd.randint(5, 60), rnd.randint(5, 60), 0.08, rnd)
    referencia = nx.bipartite.maximum_matching(a_networkx(izq, ady), top_nodes=izq)
    assert len(kuhn(izq, ady)) == len(referencia) // 2
print("Kuhn coincide con la fuerza bruta y con networkx")

Kuhn coincide con la fuerza bruta y con networkx


## 3. Hopcroft-Karp

Implementar Hopcroft-Karp: en cada fase, un BFS desde todos los vértices libres de $L$ calcula capas hasta la primera capa con un vértice libre de $R$, y un DFS que solo avanza de una capa a la siguiente encuentra un conjunto maximal de caminos aumentantes mínimos y disjuntos. Devolver el emparejamiento y el número de fases, y comprobarlo contra `kuhn`.

In [3]:
def hopcroft_karp(izquierda, adyacencia):
    pareja_izq = {u: None for u in izquierda}
    pareja_der = {}

    def capas():
        dist = {u: 0 for u in izquierda if pareja_izq[u] is None}
        cola = deque(dist)
        limite = None
        while cola:
            u = cola.popleft()
            if limite is not None and dist[u] >= limite:
                continue
            for v in adyacencia.get(u, ()):
                w = pareja_der.get(v)
                if w is None:
                    limite = dist[u] if limite is None else limite
                elif w not in dist:
                    dist[w] = dist[u] + 1
                    cola.append(w)
        return dist, limite

    def avanza(u, dist):
        for v in adyacencia.get(u, ()):
            w = pareja_der.get(v)
            if w is None or (dist.get(w) == dist[u] + 1 and avanza(w, dist)):
                pareja_izq[u], pareja_der[v] = v, u
                return True
        dist[u] = None  # callejón sin salida: no volver a entrar en esta fase
        return False

    fases = 0
    while True:
        dist, limite = capas()
        if limite is None:
            break
        fases += 1
        for u in izquierda:
            if pareja_izq[u] is None and dist.get(u) == 0:
                avanza(u, dist)
    return {u: v for u, v in pareja_izq.items() if v is not None}, fases


m, fases = hopcroft_karp(puestos, puestos)
print(m, "| fases:", fases)

for _ in range(300):
    izq, ady = aleatorio(rnd.randint(1, 30), rnd.randint(1, 30), rnd.random() * 0.3, rnd)
    m, _ = hopcroft_karp(izq, ady)
    assert es_emparejamiento(m, ady) and len(m) == len(kuhn(izq, ady))

{'Ana': 'escenario', 'Luis': 'guardarropa', 'Eva': 'taquilla', 'Pau': 'barra', 'Noa': 'megafonía'} | fases: 1


## 4. Pocas fases

La teoría asegura como mucho $2\sqrt{V}$ fases en Hopcroft-Karp. Medir las fases y el tamaño del emparejamiento en grafos aleatorios cada vez mayores y comprobar la cota.

In [4]:
print(f"{'V':>6} {'aristas':>8} {'emparejados':>12} {'fases':>6} {'2√V':>6}")
for n in (10, 100, 1000, 3000):
    izq, ady = aleatorio(n, n, 3 / n, rnd)
    m, fases = hopcroft_karp(izq, ady)
    cota = 2 * (2 * n) ** 0.5
    print(f"{2 * n:>6} {sum(map(len, ady.values())):>8} {len(m):>12} {fases:>6} {cota:>6.0f}")
    assert fases <= cota

     V  aristas  emparejados  fases    2√V
    20       26            9      3      9
   200      291           90      5     28
  2000     2971          916      8     89


  6000     8969         2777     17    155


## 5. Teorema de Hall: por qué no cabe todo el mundo

Un emparejamiento que cubre todo $L$ existe si y solo si todo subconjunto $S \subseteq L$ tiene al menos $|S|$ vecinos (condición de Hall). Cuando falta un vértice libre $u$, los vértices alcanzables desde $u$ por caminos alternantes dan un conjunto $S$ con $|N(S)| = |S| - 1$ que lo demuestra. Calcularlo para un reparto de guardias y comprobar en grafos aleatorios que se encuentra un testigo justo cuando la condición de Hall falla probando todos los subconjuntos.

In [5]:
def alcanzables(libres, adyacencia, emparejamiento):
    pareja_der = {v: u for u, v in emparejamiento.items()}
    izq, der = set(libres), set()
    cola = deque(libres)
    while cola:
        u = cola.popleft()
        for v in adyacencia.get(u, ()):
            if v not in der:
                der.add(v)
                w = pareja_der.get(v)
                if w is not None and w not in izq:
                    izq.add(w)
                    cola.append(w)
    return izq, der


def testigo_hall(izquierda, adyacencia):
    m = kuhn(izquierda, adyacencia)
    libre = next((u for u in izquierda if u not in m), None)
    return None if libre is None else alcanzables([libre], adyacencia, m)


def cumple_hall(izquierda, adyacencia):
    return all(
        len({v for u in s for v in adyacencia.get(u, ())}) >= len(s)
        for k in range(1, len(izquierda) + 1) for s in combinations(izquierda, k)
    )


guardias = {
    "Iballa": ["sábado", "domingo"],
    "Jonay": ["sábado"],
    "Tanausú": ["sábado", "domingo"],
    "Ayoze": ["viernes", "domingo"],
}
s, vecinos = testigo_hall(guardias, guardias)
print(f"S = {sorted(s)} solo puede cubrir {sorted(vecinos)}: {len(s)} personas para {len(vecinos)} días")
assert len(vecinos) == len(s) - 1 and all(set(guardias[u]) <= vecinos for u in s)

for _ in range(300):
    izq, ady = aleatorio(rnd.randint(1, 6), rnd.randint(1, 6), 0.4, rnd)
    t = testigo_hall(izq, ady)
    assert (t is None) == cumple_hall(izq, ady)
    if t:
        s, vecinos = t
        assert {v for u in s for v in ady[u]} == vecinos and len(vecinos) < len(s)

S = ['Iballa', 'Jonay', 'Tanausú'] solo puede cubrir ['domingo', 'sábado']: 3 personas para 2 días


## 6. Teorema de König: cubrimiento mínimo de vértices

Un cubrimiento de vértices es un conjunto de vértices que toca todas las aristas. Por el teorema de König, en un grafo bipartito el mínimo tiene el mismo tamaño que el emparejamiento máximo, y se construye a partir de él: con $Z$ los vértices alcanzables por caminos alternantes desde los libres de $L$, el cubrimiento es $(L \setminus Z) \cup (R \cap Z)$. Aplicarlo para elegir el mínimo de cámaras (en filas o columnas) que vigilan todas las taquillas de un recinto y comprobarlo contra la fuerza bruta.

In [6]:
def cubrimiento_minimo(izquierda, adyacencia):
    m = kuhn(izquierda, adyacencia)
    z_izq, z_der = alcanzables([u for u in izquierda if u not in m], adyacencia, m)
    return (set(izquierda) - z_izq) | z_der, m


def cubrimiento_bruto(izquierda, adyacencia):
    aristas = [(u, v) for u in izquierda for v in adyacencia.get(u, ())]
    vertices = list(izquierda) + sorted({v for _, v in aristas})
    for k in range(len(vertices) + 1):
        for c in combinations(vertices, k):
            if all(u in c or v in c for u, v in aristas):
                return k


# taquillas en (fila, columna); una cámara vigila una fila o una columna entera
taquillas = [(0, 1), (0, 4), (1, 1), (2, 0), (2, 1), (2, 2), (2, 3), (3, 1), (4, 4)]
filas = sorted({f"fila {f}" for f, _ in taquillas})
ady = {fila: [f"columna {c}" for f, c in taquillas if f"fila {f}" == fila] for fila in filas}
cubrimiento, m = cubrimiento_minimo(filas, ady)
print(f"{len(cubrimiento)} cámaras:", sorted(cubrimiento))
assert all(f"fila {f}" in cubrimiento or f"columna {c}" in cubrimiento for f, c in taquillas)
assert len(cubrimiento) == len(m) == cubrimiento_bruto(filas, ady)

for _ in range(300):
    izq, ady = aleatorio(rnd.randint(1, 5), rnd.randint(1, 5), 0.4, rnd)
    c, m = cubrimiento_minimo(izq, ady)
    assert all(u in c or v in c for u in izq for v in ady[u])
    assert len(c) == len(m) == cubrimiento_bruto(izq, ady)

3 cámaras: ['columna 1', 'columna 4', 'fila 2']


## 7. Conjunto independiente máximo

El complemento de un cubrimiento de vértices es un conjunto independiente (sin aristas entre sus vértices), así que en un grafo bipartito el independiente máximo mide $V - |M|$. Elegir el mayor grupo de alumnos y asignaturas optativas sin ninguna pareja "alumno matriculado en asignatura" (por ejemplo, para un sorteo de becas) y comprobarlo contra la fuerza bruta.

In [7]:
def independiente_maximo(izquierda, adyacencia):
    cubrimiento, _ = cubrimiento_minimo(izquierda, adyacencia)
    todos = set(izquierda) | {v for u in izquierda for v in adyacencia.get(u, ())}
    return todos - cubrimiento


matriculas = {"Ana": ["Robótica"], "Luis": ["Robótica", "Teatro"], "Eva": ["Teatro"], "Pau": ["Música"], "Noa": ["Música", "Teatro"]}
grupo = independiente_maximo(matriculas, matriculas)
print(sorted(grupo))
assert not any(v in grupo for u in grupo if u in matriculas for v in matriculas[u])

for _ in range(200):
    izq, ady = aleatorio(rnd.randint(1, 5), rnd.randint(1, 5), 0.4, rnd)
    grupo = independiente_maximo(izq, ady)
    vertices = list(izq) + sorted({v for u in izq for v in ady[u]})
    mayor = max(
        k for k in range(len(vertices) + 1) for s in combinations(vertices, k)
        if not any(u in s and v in s for u in izq for v in ady[u])
    )
    assert len(grupo) == mayor and not any(u in grupo and v in grupo for u in izq for v in ady[u])

['Ana', 'Eva', 'Luis', 'Noa', 'Pau']


## 8. Mínimo de conductores: cubrir un DAG con caminos

Una empresa de guaguas tiene viajes con hora y parada de salida y de llegada. Un conductor puede encadenar el viaje $j$ tras el $i$ si llega a tiempo a su salida. Los viajes forman un DAG y el mínimo de conductores es el mínimo de caminos disjuntos que cubren todos sus nodos: $n - |M|$, donde $M$ es el emparejamiento máximo entre una copia "salida" y una copia "llegada" de los nodos. Reconstruir los turnos de cada conductor y comprobar el mínimo contra probar todas las particiones en cadenas para pocos viajes.

In [8]:
viajes = {  # nombre: (salida, hora salida, llegada, hora llegada) en minutos
    "V1": ("Santa Cruz", 420, "La Laguna", 450),
    "V2": ("La Laguna", 460, "Tacoronte", 495),
    "V3": ("Santa Cruz", 470, "Candelaria", 505),
    "V4": ("Tacoronte", 510, "La Laguna", 545),
    "V5": ("Candelaria", 515, "Santa Cruz", 550),
    "V6": ("La Laguna", 555, "Santa Cruz", 585),
    "V7": ("Santa Cruz", 560, "La Laguna", 590),
}


def encadenables(viajes, margen=5):
    return {
        i: [j for j, (sale, hora, *_) in viajes.items() if viajes[i][2] == sale and viajes[i][3] + margen <= hora]
        for i in viajes
    }


def turnos(viajes):
    siguiente = kuhn(list(viajes), encadenables(viajes))
    tiene_anterior = set(siguiente.values())
    resultado = []
    for inicio in viajes:
        if inicio not in tiene_anterior:
            turno = [inicio]
            while turno[-1] in siguiente:
                turno.append(siguiente[turno[-1]])
            resultado.append(turno)
    return resultado


for k, turno in enumerate(turnos(viajes), 1):
    print(f"Conductor {k}:", " → ".join(f"{v} ({viajes[v][0]} {viajes[v][1] // 60}:{viajes[v][1] % 60:02})" for v in turno))


def minimo_cadenas_bruto(nodos, sucesores):
    # cada nodo elige a lo sumo un sucesor, sin repetir sucesor
    opciones = [[None, *sucesores[u]] for u in nodos]
    mejor = len(nodos)

    def prueba(k, usados):
        nonlocal mejor
        if k == len(nodos):
            mejor = min(mejor, len(nodos) - len(usados))
            return
        for v in opciones[k]:
            if v is None or v not in usados:
                prueba(k + 1, usados | ({v} if v else set()))

    prueba(0, frozenset())
    return mejor


paradas = ["A", "B", "C"]
for _ in range(100):
    aleatorios = {}
    for k in range(rnd.randint(1, 7)):
        sale = rnd.randint(0, 100)
        aleatorios[f"v{k}"] = (rnd.choice(paradas), sale, rnd.choice(paradas), sale + rnd.randint(10, 30))
    t = turnos(aleatorios)
    assert sorted(v for turno in t for v in turno) == sorted(aleatorios)
    suc = encadenables(aleatorios)
    assert all(b in suc[a] for turno in t for a, b in zip(turno, turno[1:]))
    assert len(t) == minimo_cadenas_bruto(list(aleatorios), suc)

Conductor 1: V1 (Santa Cruz 7:00) → V2 (La Laguna 7:40) → V4 (Tacoronte 8:30) → V6 (La Laguna 9:15)
Conductor 2: V3 (Santa Cruz 7:50) → V5 (Candelaria 8:35) → V7 (Santa Cruz 9:20)


## 9. Fichas de dominó en un tablero con huecos

¿Cuántas fichas de dominó $1 \times 2$ caben como máximo sin solaparse en un tablero con casillas bloqueadas? Coloreando el tablero como un ajedrez, cada ficha cubre una casilla blanca y una negra adyacentes: es un emparejamiento máximo entre blancas y negras. Resolverlo con Hopcroft-Karp, dibujar una colocación y comprobarlo con networkx en tableros aleatorios.

In [9]:
def dominos(tablero):
    libres = {(f, c) for f, fila in enumerate(tablero) for c, x in enumerate(fila) if x == "."}
    blancas = [p for p in sorted(libres) if sum(p) % 2 == 0]
    ady = {(f, c): [q for q in [(f - 1, c), (f + 1, c), (f, c - 1), (f, c + 1)] if q in libres] for f, c in blancas}
    m, _ = hopcroft_karp(blancas, ady)
    return m, blancas, ady


tablero = [
    "..#.....",
    "........",
    ".#..##..",
    "......#.",
    "#.......",
]
m, *_ = dominos(tablero)
dibujo = [list(fila) for fila in tablero]
letras = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
for k, ((f1, c1), (f2, c2)) in enumerate(m.items()):
    dibujo[f1][c1] = dibujo[f2][c2] = letras[k]
print("\n".join("".join(fila) for fila in dibujo))
print(len(m), "fichas")

for _ in range(100):
    filas, cols = rnd.randint(1, 7), rnd.randint(1, 7)
    t = ["".join("#" if rnd.random() < 0.2 else "." for _ in range(cols)) for _ in range(filas)]
    m, blancas, ady = dominos(t)
    assert len(m) == len(nx.bipartite.maximum_matching(a_networkx(blancas, ady), top_nodes=blancas)) // 2

ad#ebfcg
adiebfcg
h#il##jj
hkolmm#n
#koppqqn
17 fichas


## 10. Algoritmo húngaro

Hay que asignar $n$ guaguas a $n$ rutas, con el coste (km en vacío hasta el inicio de la ruta) de cada pareja en una matriz. Implementar el algoritmo húngaro con potenciales: para cada fila se busca, al estilo de Dijkstra, el camino aumentante de coste reducido mínimo y se ajustan los potenciales $u_i + v_j \le c_{ij}$ para que ese camino quede con coste reducido cero. Comprobar el coste contra `scipy.optimize.linear_sum_assignment` y contra probar las $n!$ permutaciones.

In [10]:
def hungaro(coste):
    """Asignación de coste mínimo para una matriz n×m con n <= m; devuelve la columna de cada fila."""
    n, m = len(coste), len(coste[0])
    infinito = float("inf")
    u, v = [0] * (n + 1), [0] * (m + 1)
    fila_de = [0] * (m + 1)  # fila_de[j]: fila (1..n) asignada a la columna j; la columna 0 es ficticia
    previa = [0] * (m + 1)
    for i in range(1, n + 1):
        fila_de[0], j0 = i, 0
        minimo = [infinito] * (m + 1)
        usada = [False] * (m + 1)
        while fila_de[j0]:
            usada[j0] = True
            i0, delta, j1 = fila_de[j0], infinito, 0
            for j in range(1, m + 1):
                if not usada[j]:
                    reducido = coste[i0 - 1][j - 1] - u[i0] - v[j]
                    if reducido < minimo[j]:
                        minimo[j], previa[j] = reducido, j0
                    if minimo[j] < delta:
                        delta, j1 = minimo[j], j
            for j in range(m + 1):
                if usada[j]:
                    u[fila_de[j]] += delta
                    v[j] -= delta
                else:
                    minimo[j] -= delta
            j0 = j1
        while j0:
            j1 = previa[j0]
            fila_de[j0] = fila_de[j1]
            j0 = j1
    asignacion = [0] * n
    for j in range(1, m + 1):
        if fila_de[j]:
            asignacion[fila_de[j] - 1] = j - 1
    return asignacion


guaguas = ["G-101", "G-102", "G-103", "G-104"]
rutas = ["Anaga", "Teno", "Teide", "Sur"]
km = [
    [10, 12, 30, 50],
    [11, 30, 25, 38],
    [28, 31, 15, 20],
    [55, 35, 16, 18],
]
asignacion = hungaro(km)
for g, j in zip(guaguas, asignacion):
    print(f"{g} → {rutas[j]:6} {km[guaguas.index(g)][j]:>3} km")
total = sum(km[i][j] for i, j in enumerate(asignacion))
filas, columnas = linear_sum_assignment(km)
print("Total:", total)
assert total == np.array(km)[filas, columnas].sum()

rng = np.random.default_rng(8)
for _ in range(300):
    n = int(rng.integers(1, 7))
    c = rng.integers(0, 50, size=(n, n)).tolist()
    a = hungaro(c)
    assert sorted(a) == list(range(n))
    assert sum(c[i][a[i]] for i in range(n)) == min(sum(c[i][p[i]] for i in range(n)) for p in permutations(range(n)))

G-101 → Teno    12 km
G-102 → Anaga   11 km
G-103 → Teide   15 km
G-104 → Sur     18 km
Total: 56


## 11. Matrices rectangulares y maximizar

Ampliar el ejercicio anterior: con más filas que columnas se trabaja con la traspuesta, y para maximizar (por ejemplo, la puntuación de cada voluntario en cada tarea) basta restar cada valor del máximo. Comprobarlo contra `linear_sum_assignment` con y sin `maximize=True` en matrices aleatorias de cualquier forma, con valores decimales.

In [11]:
def asignacion_optima(matriz, maximizar=False):
    c = np.asarray(matriz, dtype=float)
    if maximizar:
        c = c.max() - c
    if c.shape[0] <= c.shape[1]:
        return list(enumerate(hungaro(c.tolist())))
    return sorted((i, j) for j, i in enumerate(hungaro(c.T.tolist())))


puntuacion = [
    [7, 3, 9],
    [6, 8, 4],
    [5, 9, 6],
    [8, 2, 7],
    [4, 6, 5],
]
voluntarios = ["Ana", "Luis", "Eva", "Pau", "Noa"]
tareas = ["cocina", "sonido", "puerta"]
parejas = asignacion_optima(puntuacion, maximizar=True)
print({voluntarios[i]: tareas[j] for i, j in parejas}, "puntuación", sum(puntuacion[i][j] for i, j in parejas))

for _ in range(300):
    c = rng.random(size=rng.integers(1, 8, size=2)) * 100
    for maximizar in (False, True):
        parejas = asignacion_optima(c, maximizar)
        f, k = linear_sum_assignment(c, maximize=maximizar)
        assert len(parejas) == len(f) == min(c.shape)
        assert np.isclose(sum(c[i, j] for i, j in parejas), c[f, k].sum())

{'Ana': 'puerta', 'Eva': 'sonido', 'Pau': 'cocina'} puntuación 26
